# RQ4 - Scalability

Plots wall-clock time, event throughput, memory usage, and trace size.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'RQ4').exists() and (ROOT / 'Paper Results').exists():
    ROOT = ROOT / 'Paper Results'

rq4 = ROOT / 'RQ4'
summary_path = rq4 / 'scalability_summary.csv'

if not summary_path.exists():
    print(f'RQ4 data not found: {summary_path}')
    print('Run: python "Paper Results/run_rq4_experiments.py"')
else:
    df = pd.read_csv(summary_path)
    for col in df.columns:
        if col not in ['coord_label', 'architecture']:
            df[col] = pd.to_numeric(df[col], errors='ignore')

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    wall = df.groupby(['coord_label', 'sim_days'])['wall_time_s_mean'].mean().reset_index()
    for coord, data in wall.groupby('coord_label'):
        ax.plot(data['sim_days'], data['wall_time_s_mean'], marker='o', label=coord)
    ax.set_title('Wall-clock time vs simulation duration')
    ax.set_xlabel('Simulation days')
    ax.set_ylabel('Seconds')
    ax.grid(alpha=0.25)
    ax.legend()

    ax = axes[0, 1]
    rss = df.groupby(['coord_label', 'n_robots'])['max_rss_mb_mean'].mean().reset_index()
    for coord, data in rss.groupby('coord_label'):
        ax.plot(data['n_robots'], data['max_rss_mb_mean'], marker='o', label=coord)
    ax.set_title('Peak RSS vs fleet size')
    ax.set_xlabel('Agents')
    ax.set_ylabel('MB')
    ax.grid(alpha=0.25)
    ax.legend()

    ax = axes[1, 0]
    trace = df.groupby(['coord_label', 'item_spawning_time'])['total_output_size_mb_mean'].mean().reset_index()
    for coord, data in trace.groupby('coord_label'):
        data = data.sort_values('item_spawning_time')
        ax.plot(data['item_spawning_time'], data['total_output_size_mb_mean'], marker='o', label=coord)
    ax.invert_xaxis()
    ax.set_title('Trace size vs workload intensity')
    ax.set_xlabel('Mean spawn interval (s, lower means higher load)')
    ax.set_ylabel('MB')
    ax.grid(alpha=0.25)
    ax.legend()

    ax = axes[1, 1]
    throughput = df.groupby(['coord_label', 'n_robots'])['events_per_wall_s_mean'].mean().reset_index()
    for coord, data in throughput.groupby('coord_label'):
        ax.plot(data['n_robots'], data['events_per_wall_s_mean'], marker='o', label=coord)
    ax.set_title('Simulator throughput')
    ax.set_xlabel('Agents')
    ax.set_ylabel('Events per wall-clock second')
    ax.grid(alpha=0.25)
    ax.legend()

    fig.suptitle('RQ4: Practical scalability limits', fontsize=14)
    fig.tight_layout()
    plt.show()
    display(df)
